In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.cuda.amp import autocast, GradScaler
import psutil
import warnings
import gc

# 1. OPTIMIZE MEMORY ALLOCATION
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Suppress warnings
warnings.filterwarnings('ignore')

# ==============================
# 2. CONFIGURATION (Safe Mode)
# ==============================
class EXTREME_CONFIG:
    # --- Paths ---
    # These match the standard RunPod setup script
    train_images_dir = "dataset/train_images/forged"
    train_masks_dir = "dataset/train_masks"
    authentic_images_dir = "dataset/train_images/authentic"
    
    # --- Model ---
    model_name = "facebook/dinov2-large" # 1 Billion Parameters
    
    # --- Hardware Settings (2x A40) ---
    img_size = 1024          # Full HD Training
    
    # BATCH SIZE FIX: 12 crashed. 6 is safe.
    # 6 images * 2 GPUs = 12 Physical Batch.
    batch_size = 6           
    
    # ACCUMULATION: We accumulate 4 steps to simulate a huge batch.
    # 12 Physical * 4 Steps = Effective Batch Size 48.
    accumulation_steps = 4   
    
    epochs = 40              
    lr = 2e-5                
    
    save_path = "dinov2_large_a40_extreme.pth"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_workers = 16         

# ==============================
# 3. RAM-CACHED DATASET
# ==============================
class RAMCachedDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.transform = transform
        self.data_cache = []
        
        print(f"🧠 Caching {len(image_paths)} images into RAM...")
        
        for img_path, mask_path in tqdm(zip(image_paths, mask_paths), total=len(image_paths)):
            # Read Image
            img = cv2.imread(img_path)
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Read Mask (Support .npy and .png)
            if mask_path is None:
                mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
            elif str(mask_path).endswith('.npy'):
                mask = np.load(mask_path)
                if len(mask.shape) == 3: mask = np.max(mask, axis=0)
                mask = (mask > 0).astype(np.uint8)
            else:
                # Fallback for PNG masks
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if mask is None:
                    mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
                else:
                    mask = (mask > 0).astype(np.uint8)
                
            self.data_cache.append((img, mask))
            
        mem_usage = psutil.virtual_memory().percent
        print(f"✅ Cache Complete! System RAM Used: {mem_usage}%")

    def __len__(self):
        return len(self.data_cache)

    def __getitem__(self, idx):
        img, mask = self.data_cache[idx]
        mask = mask.astype(np.float32)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask'].unsqueeze(0)
            
        return img, mask

def get_transforms(phase):
    if phase == 'train':
        return A.Compose([
            A.Resize(EXTREME_CONFIG.img_size, EXTREME_CONFIG.img_size),
            
            # Geometric Augmentations
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=45, p=0.5),
            
            # Forgery Specific
            A.ImageCompression(quality_range=(40, 100), p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),
            A.RGBShift(r_shift_limit=20, g_shift_limit=20, b_shift_limit=20, p=0.3),
            A.GaussianBlur(blur_limit=(3, 7), p=0.2),
            
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(EXTREME_CONFIG.img_size, EXTREME_CONFIG.img_size),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

# ==============================
# 4. DINOv2 LARGE MODEL
# ==============================
class DinoLargeSegmenter(nn.Module):
    def __init__(self):
        super().__init__()
        print(f"Loading {EXTREME_CONFIG.model_name}...")
        self.encoder = AutoModel.from_pretrained(EXTREME_CONFIG.model_name)
        self.embed_dim = 1024 
        
        self.decoder = nn.Sequential(
            nn.Conv2d(self.embed_dim, 512, 3, padding=1), 
            nn.BatchNorm2d(512), nn.ReLU(),
            nn.Dropout(0.1), # Prevent overfitting
            
            nn.Conv2d(512, 256, 3, padding=1), 
            nn.BatchNorm2d(256), nn.ReLU(),
            
            nn.Conv2d(256, 64, 3, padding=1), 
            nn.BatchNorm2d(64), nn.ReLU(),
            
            nn.Conv2d(64, 1, 1)
        )
        
    def forward(self, x):
        outputs = self.encoder(pixel_values=x)
        last_hidden_state = outputs.last_hidden_state
        B, N, C = last_hidden_state.shape
        H_feat = int(np.sqrt(N-1))
        
        features = last_hidden_state[:, 1:, :].permute(0, 2, 1).reshape(B, C, H_feat, H_feat)
        masks = self.decoder(features)
        masks = F.interpolate(masks, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        return masks

# ==============================
# 5. TRAINING ENGINE
# ==============================
def train():
    print("📂 Loading Dataset Paths...")
    
    # Path Fallback Logic
    if not os.path.exists(EXTREME_CONFIG.train_images_dir):
        if os.path.exists("dataset/train_images/forged"):
            pass # Paths are correct
        elif os.path.exists("train_images/forged"):
            # Update paths to match local root
            EXTREME_CONFIG.train_images_dir = "train_images/forged"
            EXTREME_CONFIG.train_masks_dir = "train_masks"
            EXTREME_CONFIG.authentic_images_dir = "train_images/authentic"
        else:
            raise FileNotFoundError("Could not find dataset folders! Did you run the setup script?")

    # 1. Gather Forged
    forged_files = sorted([os.path.join(EXTREME_CONFIG.train_images_dir, f) for f in os.listdir(EXTREME_CONFIG.train_images_dir)])
    img_names = [os.path.splitext(os.path.basename(f))[0] for f in forged_files]
    
    valid_imgs, valid_masks = [], []
    
    # 2. Match Masks (Robust .npy / .png check)
    print("   Matching masks...")
    for img, name in zip(forged_files, img_names):
        npy_path = os.path.join(EXTREME_CONFIG.train_masks_dir, name + ".npy")
        png_path = os.path.join(EXTREME_CONFIG.train_masks_dir, name + ".png")
        
        if os.path.exists(npy_path):
            valid_imgs.append(img)
            valid_masks.append(npy_path)
        elif os.path.exists(png_path):
            valid_imgs.append(img)
            `!`
            valid_masks.append(png_path)
            
    # 3. Add Authentic
    auth_files = sorted([os.path.join(EXTREME_CONFIG.authentic_images_dir, f) for f in os.listdir(EXTREME_CONFIG.authentic_images_dir)])
    # Use 3500 authentic images (High RAM allows this)
    num_auth = min(len(auth_files), 3500)
    auth_subset = auth_files[:num_auth]
    ghjkjhghjuikoliugfghyuioiuhygt
    7yuhjio98uyhjkio8uytghjnkliou
    ghjbnvghyujn
    valid_imgs.extend(auth_subset)
    valid_masks.extend([None] * num_auth)
    
    print(f"   Total Training Images: {len(valid_imgs)}")
    
    # 4. Split
    train_x, val_x, train_y, val_y = train_test_split(valid_imgs, valid_masks, test_size=0.1, random_state=42)
    
    # 5. Initialize Caches
    print("🚀 Initializing RAM Cache...")
    train_ds = RAMCachedDataset(train_x, train_y, get_transforms('train'))
    val_ds = RAMCachedDataset(val_x, val_y, get_transforms('valid'))
    
    train_loader = DataLoader(train_ds, batch_size=EXTREME_CONFIG.batch_size * torch.cuda.device_count(), 
                              shuffle=True, num_workers=EXTREME_CONFIG.num_workers, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=EXTREME_CONFIG.batch_size * torch.cuda.device_count(), 
                            shuffle=False, num_workers=EXTREME_CONFIG.num_workers)

    # 6. Model Setup
    print(f"🔥 Initializing Model on {torch.cuda.device_count()} A40 GPUs...")
    model = DinoLargeSegmenter()
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(EXTREME_CONFIG.device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=EXTREME_CONFIG.lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    scaler = GradScaler()
    
    best_dice = 0.0

    # 7. Training Loop (With Gradient Accumulation)
    for epoch in range(EXTREME_CONFIG.epochs):
        model.train()
        train_loss = 0
        optimizer.zero_grad()
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        
        for step, (imgs, masks) in enumerate(pbar):
            imgs, masks = imgs.to(EXTREME_CONFIG.device), masks.to(EXTREME_CONFIG.device)
            
            with autocast():
                preds = model(imgs)
                loss = criterion(preds, masks)
                # Normalize loss for accumulation
                loss = loss / EXTREME_CONFIG.accumulation_steps
            
            scaler.scale(loss).backward()
            
            # Step only after N steps
            if (step + 1) % EXTREME_CONFIG.accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            train_loss += loss.item() * EXTREME_CONFIG.accumulation_steps
            pbar.set_postfix(loss=loss.item() * EXTREME_CONFIG.accumulation_steps)
            
        # Validation
        model.eval()
        val_dice = 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(EXTREME_CONFIG.device), masks.to(EXTREME_CONFIG.device)
                with autocast():
                    preds = model(imgs)
                pred_mask = (torch.sigmoid(preds) > 0.5).float()
                inter = (pred_mask * masks).sum()
                dice = (2. * inter) / (pred_mask.sum() + masks.sum() + 1e-6)
                val_dice += dice.item()
        
        val_dice /= len(val_loader)
        print(f"Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | Val Dice: {val_dice:.4f}")
        
        if val_dice > best_dice:
            print(f"🏆 Best Score: {val_dice:.4f}")
            best_dice = val_dice
            state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(state_dict, EXTREME_CONFIG.save_path)
            
        # GC to keep memory clean
        gc.collect()
        torch.cuda.empty_cache()

if __name__ == "__main__":
    try:
        train()
    except Exception as e:
        print(f"❌ Error: {e}")

📂 Loading Dataset Paths...
   Matching masks...
   Total Training Images: 5128
🚀 Initializing RAM Cache...
🧠 Caching 4615 images into RAM...


  0%|          | 0/4615 [00:00<?, ?it/s]

✅ Cache Complete! System RAM Used: 22.6%
🧠 Caching 513 images into RAM...


  0%|          | 0/513 [00:00<?, ?it/s]

✅ Cache Complete! System RAM Used: 23.4%
🔥 Initializing Model on 2 A40 GPUs...
Loading facebook/dinov2-large...


...

Epoch 18:   0%|          | 0/384 [00:07<?, ?it/s]

Epoch 18 | Loss: 0.2708 | Val Dice: 0.0028


Epoch 19:   0%|          | 0/384 [00:07<?, ?it/s]

Epoch 19 | Loss: 0.2630 | Val Dice: 0.0000


Epoch 20:   0%|          | 0/384 [00:08<?, ?it/s]